**1: Preparação do Ambiente**

In [ ]:
# Célula 1: Setup do Ambiente de Avaliação
from google.colab import drive
drive.mount('/content/drive')

# Instalação das dependências para CLIPScore, Gradio e Hugging Face
!pip install -q diffusers transformers accelerate torchmetrics[image] gradio gTTS huggingface_hub
print("✅ Ambiente preparado com sucesso!")

In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install -U torchmetrics

**2: (Etapa 3) — Avaliação Matemática (CLIPScore)**

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Carrega os pipelines
# -----------------------------
print("Carregando modelo base...")
pipe_base = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(device)

print("Carregando LoRA...")
pipe_lora = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(device)

pipe_lora.load_lora_weights(
    "/content/drive/MyDrive/Atelie_Generativo_LoRA/PixelArt_Rank32"
)

# -----------------------------
# Carrega o CLIP
# -----------------------------
print("Carregando CLIP...")

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch16"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch16"
)

# -----------------------------
# Prompt de avaliação
# -----------------------------
prompt = "estilo_pixel_art, plantação cercada com fileiras de milho maduro em terreno agrícola."

# Seed idêntica
generator = torch.Generator(device=device).manual_seed(42)

img_base = pipe_base(
    prompt,
    num_inference_steps=30,
    generator=generator
).images[0]

generator = torch.Generator(device=device).manual_seed(42)

img_lora = pipe_lora(
    prompt,
    num_inference_steps=30,
    generator=generator
).images[0]

# -----------------------------
# Calcula similaridade CLIP
# -----------------------------
@torch.no_grad()
def clip_score(image, text):

    inputs = clip_processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = clip_model(**inputs)

    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

    similarity = (image_embeds @ text_embeds.T).item()

    return similarity

score_base = clip_score(img_base, prompt)
score_lora = clip_score(img_lora, prompt)

print(f"CLIP Similarity Base : {score_base:.4f}")
print(f"CLIP Similarity LoRA : {score_lora:.4f}")

# -----------------------------
# Exibe imagens
# -----------------------------
fig, ax = plt.subplots(1,2,figsize=(13,6))

ax[0].imshow(img_base)
ax[0].set_title(f"Base\nCLIP={score_base:.4f}")
ax[0].axis("off")

ax[1].imshow(img_lora)
ax[1].set_title(f"LoRA\nCLIP={score_lora:.4f}")
ax[1].axis("off")

plt.show()

👨‍🏫 Avaliação Humana e Overfitting:
> Overfitting: O modelo decorou os dados ou aprendeu o estilo? Analisem a imagem gerada pelo LoRA. Se ela for idêntica a uma das fotos do dataset, houve memorização (overfitting). Se for uma cena nova, mas com a "vibe" do Pixel Art, houve generalização bem-sucedida!



**3: O "Shift" para Cloud (Subindo o modelo para o Hugging Face Hub)**

In [ ]:
# Célula 3: Publicação do Modelo (Weights) no Hugging Face Hub
from huggingface_hub import HfApi, login

print("🔑 Faça o login com o seu Token de ESCRITA (Write Token) do Hugging Face:")
login() # Um prompt aparecerá para você colar o token (não escreva no código!)

# Substitua pelas suas credenciais
SEU_USUARIO_HF = "Thiagofariasl" # Coloque seu usuário real aqui
NOME_DO_MODELO = "pixelart-16bit-lora"

api = HfApi()
repo_id = f"{SEU_USUARIO_HF}/{NOME_DO_MODELO}"

print(f"🚀 Criando repositório público: {repo_id}...")
api.create_repo(repo_id=repo_id, exist_ok=True)

caminho_lora = "/content/drive/MyDrive/Atelie_Generativo_LoRA/PixelArt_Rank8"

print("⏳ Enviando os pesos do LoRA. Isso pode levar alguns minutos...")
api.upload_folder(
    folder_path=caminho_lora, # A pasta do Drive com o rank32
    repo_id=repo_id,
    repo_type="model"
)
print("✅ LoRA publicado com sucesso no Hugging Face Hub!")

**4: (Etapa 4) — Desenvolvimento do Pipeline Multimodal Web**

In [ ]:
# Célula 4: Construção da Estrutura do App (Infraestrutura as Code)
import os

app_dir = "/content/meu_app_espaco"
os.makedirs(app_dir, exist_ok=True)

# 1. Criação do requirements.txt
with open(os.path.join(app_dir, "requirements.txt"), "w") as f:
    f.write("gradio\ndiffusers\ntransformers\naccelerate\ntorch\ngTTS\npeft\nsafetensors\n")

# 2. Criação do app.py (O Pipeline Multimodal)
app_code = """
import gradio as gr
import torch
from diffusers import StableDiffusionPipeline
from gtts import gTTS
import os

# Configuração dinâmica: Usa GPU se disponível no Spaces (ex: ZeroGPU), senão cai para CPU (Inference via software)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Inicializando modelo no dispositivo: {device}")
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch_dtype
).to(device)

# --- ALERTA: SUBSTITUAM PELA REPO DE VOCÊS AQUI ---
# Este comando usa a Inference API/Hub para baixar o LoRA de vocês dinamicamente!
pipe.load_lora_weights("Seu_nome/pixelart-16bit-lora")

def gerar_experiencia_multimodal(prompt_usuario):
    # Modalidade 1: Processamento de Texto (Orquestração do Prompt)
    prompt_orquestrado = f"estilo_pixel_art, {prompt_usuario}, cena em pixel art com cabana rural, rio, píer, horta cultivada e personagem de um agricultor."

    # Modalidade 2: Geração de Imagem (Visão Computacional)
    # Reduzimos os steps para 20 para agilizar caso rode em CPU no Hugging Face
    imagem = pipe(prompt_orquestrado, num_inference_steps=20, guidance_scale=7.5).images[0]

    # Modalidade 3: Síntese de Voz (Áudio)
    texto_narracao = f"Gerando em pixel art para a sua solicitação: {prompt_usuario}."
    tts = gTTS(texto_narracao, lang='pt')
    audio_path = "narracao.mp3"
    tts.save(audio_path)

    return prompt_orquestrado, imagem, audio_path

# Interface UI/UX (Gradio Blocks)
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# 👾 Ateliê Generativo: Multimodal Pixel Art (16-bits)")
    gr.Markdown("Transforme suas ideias em arte retrô narrada. Pipeline Integrado: Texto ➔ Imagem ➔ Áudio.")

    with gr.Row():
        with gr.Column(scale=1):
            entrada = gr.Textbox(label="O que você deseja criar?", placeholder="Ex: cena em pixel art com linda cabana rural...")
            btn = gr.Button("Sintetizar", variant="primary")
            saida_prompt = gr.Textbox(label="Texto Orquestrado", interactive=False)
            saida_audio = gr.Audio(label="Voz do Sistema (Áudio)")

        with gr.Column(scale=2):
            saida_imagem = gr.Image(label="Arte Gerada (Seu LoRA)")

    btn.click(fn=gerar_experiencia_multimodal, inputs=entrada, outputs=[saida_prompt, saida_imagem, saida_audio])

demo.launch()
"""

with open(os.path.join(app_dir, "app.py"), "w", encoding="utf-8") as f:
    f.write(app_code.replace("Seu_nome/pixelart-16bit-lora", f"{SEU_USUARIO_HF}/{NOME_DO_MODELO}"))

print("✅ Arquivos `app.py` e `requirements.txt` gerados com sucesso na pasta /content/meu_app_espaco!")

**5: Deploy Final no Hugging Face Spaces (O "Go Live")**

**⚠️ Gestão de Chaves e Segurança:**
> Como estamos submetendo via API e usando as variáveis nativas do Hugging Face, os tokens estão criptografados.



In [ ]:
# Célula 5: Deploy da Aplicação no Hugging Face Spaces
from huggingface_hub import HfApi

api = HfApi()
repo_space_id = f"{SEU_USUARIO_HF}/atelie-multimodal-pixelart"

print(f"🚀 Criando o Servidor (Space) público: {repo_space_id}...")
# Criamos um Space com o SDK do Gradio
api.create_repo(repo_id=repo_space_id, repo_type="space", space_sdk="gradio", exist_ok=True)

print("⏳ Fazendo o Deploy dos arquivos da aplicação. Aguarde...")
api.upload_folder(
    folder_path=app_dir,
    repo_id=repo_space_id,
    repo_type="space"
)

print(f"🎉 DEPLOY CONCLUÍDO! Acesse sua aplicação web em: https://huggingface.co/spaces/{repo_space_id}")

🏆 Validação do Resultado

> Após executar a Célula 5, clique no link gerado e vá para a aba Files and versions no seu Space para conferir se o app.py está lá. O Space levará alguns minutos no estado Building (Construindo a imagem Docker). Assim que ficar Running, o seu pipeline multimodal estará acessível para avaliação!

